In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")

connected


In [2]:
con.execute("""
CREATE OR REPLACE TABLE repeat_summary AS
WITH per_customer AS (
  SELECT customer_unique_id, COUNT(*) AS n_orders
  FROM order_level
  WHERE order_status = 'delivered'
  GROUP BY 1
)
SELECT COUNT(*) AS customers,
       SUM(CASE WHEN n_orders >= 2 THEN 1 ELSE 0 END) AS repeat_customers,
       ROUND(100.0 * SUM(CASE WHEN n_orders >= 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS repeat_pct,
       ROUND(AVG(n_orders), 3) AS avg_orders_per_customer
FROM per_customer
""")

con.execute("SELECT * FROM repeat_summary").df()

,customers,repeat_customers,repeat_pct,avg_orders_per_customer
0,93358,2801.0,3.0,1.033


In [3]:
con.execute("""
CREATE OR REPLACE TABLE cohort_retention AS
WITH delivered AS (
  SELECT customer_unique_id,
         DATE_TRUNC('month', order_purchase_timestamp) AS order_month
  FROM order_level
  WHERE order_status = 'delivered'
),
first_order AS (
  SELECT customer_unique_id, MIN(order_month) AS cohort_month
  FROM delivered
  GROUP BY 1
),
activity AS (
  SELECT DISTINCT customer_unique_id, order_month
  FROM delivered
),
counts AS (
  SELECT f.cohort_month,
         DATE_DIFF('month', f.cohort_month, a.order_month) AS months_since,
         COUNT(DISTINCT a.customer_unique_id) AS customers
  FROM first_order f
  JOIN activity a USING (customer_unique_id)
  GROUP BY 1, 2
),
sizes AS (
  SELECT cohort_month, customers AS cohort_size
  FROM counts
  WHERE months_since = 0
),
grid AS (
  SELECT s.cohort_month, s.cohort_size, m.months_since
  FROM sizes s
  CROSS JOIN (SELECT UNNEST(range(0, 20)) AS months_since) m
  WHERE s.cohort_month BETWEEN DATE '2017-01-01' AND DATE '2018-08-01'
    AND m.months_since <= DATE_DIFF('month', s.cohort_month, DATE '2018-08-01')
)
SELECT g.cohort_month,
       g.months_since,
       COALESCE(c.customers, 0) AS customers,
       g.cohort_size,
       ROUND(100.0 * COALESCE(c.customers, 0) / g.cohort_size, 2) AS retention_pct
FROM grid g
LEFT JOIN counts c
  ON c.cohort_month = g.cohort_month AND c.months_since = g.months_since
ORDER BY 1, 2
""")

con.execute("SELECT * FROM cohort_retention LIMIT 10").df()

,cohort_month,months_since,customers,cohort_size,retention_pct
0,2017-01-01,0,717,717,100.00
1,2017-01-01,1,2,717,0.28
2,2017-01-01,2,2,717,0.28
3,2017-01-01,3,1,717,0.14
4,2017-01-01,4,3,717,0.42
5,2017-01-01,5,1,717,0.14
6,2017-01-01,6,3,717,0.42
7,2017-01-01,7,1,717,0.14
8,2017-01-01,8,1,717,0.14
9,2017-01-01,9,0,717,0.00


In [4]:
con.execute("""
CREATE OR REPLACE TABLE avg_retention AS
SELECT months_since,
       SUM(customers) AS customers,
       SUM(cohort_size) AS cohort_size_total,
       ROUND(100.0 * SUM(customers) / SUM(cohort_size), 2) AS retention_pct
FROM cohort_retention
WHERE months_since <= 6
GROUP BY 1
ORDER BY 1
""")

con.execute("SELECT * FROM avg_retention").df()

,months_since,customers,cohort_size_total,retention_pct
0,0,93094.0,93094.0,100.00
1,1,420.0,86950.0,0.48
2,2,273.0,81001.0,0.34
3,3,192.0,75123.0,0.26
4,4,176.0,68617.0,0.26
5,5,140.0,62035.0,0.23
6,6,128.0,55261.0,0.23


In [5]:
df = con.execute("SELECT cohort_month, months_since, retention_pct FROM cohort_retention").df()
df.pivot(index="cohort_month", columns="months_since", values="retention_pct").round(2)

months_since,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
cohort_month,,,,,,,,,,,,,,,,,,,,
2017-01-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14,0.14,0.00,0.42,0.14,0.70,0.42,0.14,0.14,0.28,0.42,0.0,0.14
2017-02-01,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18,0.12,0.18,0.12,0.31,0.12,0.18,0.12,0.06,0.06,0.18,0.0,NaN
2017-03-01,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32,0.32,0.08,0.36,0.12,0.20,0.12,0.16,0.24,0.08,0.12,NaN,NaN
2017-04-01,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31,0.31,0.18,0.27,0.09,0.04,0.04,0.09,0.09,0.13,NaN,NaN,NaN
2017-05-01,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14,0.26,0.26,0.26,0.35,0.23,0.03,0.17,0.20,NaN,NaN,NaN,NaN
2017-06-01,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23,0.13,0.20,0.30,0.36,0.16,0.16,0.23,NaN,NaN,NaN,NaN,NaN
2017-07-01,100.0,0.53,0.35,0.24,0.29,0.21,0.32,0.11,0.19,0.27,0.21,0.29,0.13,0.24,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-01,100.0,0.69,0.35,0.27,0.35,0.52,0.30,0.27,0.15,0.15,0.25,0.20,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,100.0,0.70,0.55,0.27,0.45,0.22,0.22,0.25,0.27,0.17,0.25,0.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
con.execute("COPY cohort_retention TO 'dashboard/cohort_retention.csv' (HEADER, DELIMITER ',')")
con.execute("COPY avg_retention TO 'dashboard/avg_retention.csv' (HEADER, DELIMITER ',')")
con.execute("COPY repeat_summary TO 'dashboard/repeat_summary.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [7]:
r = con.execute("SELECT * FROM repeat_summary").df().iloc[0]
a = con.execute("SELECT * FROM avg_retention").df().set_index("months_since")

In [8]:
def ret(k):
    return f"{a.loc[k, 'retention_pct']}%" if k in a.index else "n/a"

if r.repeat_pct < 10:
    verdict = "Repeat purchase is very low, so growth depends on acquiring new customers rather than on loyalty."
else:
    verdict = "Repeat purchase is higher than expected, so review the retention table before drawing conclusions."

In [9]:
print("=== COPY THIS INTO business_questions.md (Q3 answer) ===")
print(f"- Answer: Of {int(r.customers):,} customers with a delivered order, {int(r.repeat_customers):,} ({r.repeat_pct}%) ordered more than once, with an average of {r.avg_orders_per_customer} orders per customer. "
      f"Averaged across the Jan 2017 to Aug 2018 cohorts, {ret(1)} of customers bought again one month after their first order, {ret(3)} after three months and {ret(6)} after six months. "
      f"{verdict} "
      f"Later cohorts have fewer months of history, so their retention can only be measured over a shorter period.")
print()
print("=== COPY THIS INTO README.md (under ## Method) ===")
print("- Customers are identified by customer_unique_id, not customer_id, because customer_id changes with every order. Cohort retention uses delivered orders and cohorts from Jan 2017 to Aug 2018, where a cohort is the month of a customer's first delivered order. Repeat orders placed in the same month as the first order count as month 0, not as retention.")

=== COPY THIS INTO business_questions.md (Q3 answer) ===
- Answer: Of 93,358 customers with a delivered order, 2,801 (3.0%) ordered more than once, with an average of 1.033 orders per customer. Averaged across the Jan 2017 to Aug 2018 cohorts, 0.48% of customers bought again one month after their first order, 0.26% after three months and 0.23% after six months. Repeat purchase is very low, so growth depends on acquiring new customers rather than on loyalty. Later cohorts have fewer months of history, so their retention can only be measured over a shorter period.

=== COPY THIS INTO README.md (under ## Method) ===
- Customers are identified by customer_unique_id, not customer_id, because customer_id changes with every order. Cohort retention uses delivered orders and cohorts from Jan 2017 to Aug 2018, where a cohort is the month of a customer's first delivered order. Repeat orders placed in the same month as the first order count as month 0, not as retention.


In [10]:
con.close()
print("closed")

closed
